# Olist E-Commerce SQL Case Study

A pure-SQL analysis of the Brazilian E-Commerce Public Dataset by Olist, exploring revenue trends, delivery performance, customer retention, and seller rankings across a multi-table relational database.

**Database:** olist.db (SQLite), built from 8 source CSVs
**Tables:** orders, order_items, order_payments, order_reviews, customers, products, sellers, category_translation

In [1]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect("olist.db")

csv_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table_name, file_name in csv_files.items():
    df = pd.read_csv(file_name)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {df.shape[0]} rows")

conn.close()
print("Database created: olist.db")

Loaded customers: 99441 rows
Loaded orders: 99441 rows
Loaded order_items: 112650 rows
Loaded order_payments: 103886 rows
Loaded order_reviews: 99224 rows
Loaded products: 32951 rows
Loaded sellers: 3095 rows
Loaded category_translation: 71 rows
Database created: olist.db


In [5]:
conn = sqlite3.connect("olist.db")

In [7]:
query = """
SELECT *
FROM orders
LIMIT 5;
"""

pd.read_sql(query, conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 1. Order status breakdown
What does the overall order pipeline look like?

In [9]:
query = """
SELECT
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
"""

pd.read_sql(query, conn)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 2. Total revenue and average order value
What is total revenue, and what's the typical order size?

In [11]:
query = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
"""

pd.read_sql(query, conn)

,total_orders,total_revenue,avg_order_value
0,96478,13221498.11,137.04


## 3. Monthly revenue trend
How does revenue trend over time?

In [13]:
query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;
"""

pd.read_sql(query, conn)

,order_month,total_orders,total_revenue
0,2016-09,1,134.97
1,2016-10,265,40325.11
2,2016-12,1,10.90
3,2017-01,750,111798.36
4,2017-02,1653,234223.40
5,2017-03,2546,359198.85
6,2017-04,2303,340669.68
7,2017-05,3546,489338.25
8,2017-06,3135,421923.37
9,2017-07,3872,481604.52


## 4. Top 10 product categories by revenue
Which product categories drive the most revenue?

In [15]:
query = """
SELECT
    ct.product_category_name_english AS category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,category,total_orders,total_revenue
0,health_beauty,8647,1233131.72
1,watches_gifts,5495,1166176.98
2,bed_bath_table,9272,1023434.76
3,sports_leisure,7530,954852.55
4,computers_accessories,6530,888724.61
5,furniture_decor,6307,711927.69
6,housewares,5743,615628.69
7,cool_stuff,3559,610204.10
8,auto,3810,578966.65
9,toys,3804,471286.48


## 5. Late delivery rate
What percentage of delivered orders arrived late?

In [17]:
query = """
SELECT
    COUNT(*) AS total_delivered_orders,
    SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_pct
FROM orders
WHERE order_status = 'delivered';
"""

pd.read_sql(query, conn)

,total_delivered_orders,late_orders,late_pct
0,96478,7826,8.11


## 6. Delivery lateness vs. review score
Do late deliveries hurt customer satisfaction?

In [19]:
query = """
SELECT
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Late'
        ELSE 'On Time'
    END AS delivery_status,
    COUNT(*) AS num_orders,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY delivery_status;
"""

pd.read_sql(query, conn)

,delivery_status,num_orders,avg_review_score
0,Late,7700,2.57
1,On Time,88661,4.29


## 7. Revenue by customer state
Which states generate the most revenue?

In [21]:
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_state,total_orders,total_revenue,avg_order_value
0,SP,40501,5067633.16,125.12
1,RJ,12350,1759651.13,142.48
2,MG,11354,1552481.83,136.73
3,RS,5345,728897.47,136.37
4,PR,4923,666063.51,135.30
5,SC,3546,507012.13,142.98
6,BA,3256,493584.14,151.59
7,DF,2080,296498.41,142.55
8,GO,1957,282836.70,144.53
9,ES,1995,268643.45,134.66


## 8. Cumulative monthly revenue
What does the cumulative growth trajectory look like?

In [23]:
query = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
        ROUND(SUM(oi.price), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY order_month
)
SELECT
    order_month,
    revenue,
    ROUND(SUM(revenue) OVER (ORDER BY order_month), 2) AS cumulative_revenue
FROM monthly_revenue
ORDER BY order_month;
"""

pd.read_sql(query, conn)

,order_month,revenue,cumulative_revenue
0,2016-09,134.97,134.97
1,2016-10,40325.11,40460.08
2,2016-12,10.90,40470.98
3,2017-01,111798.36,152269.34
4,2017-02,234223.40,386492.74
5,2017-03,359198.85,745691.59
6,2017-04,340669.68,1086361.27
7,2017-05,489338.25,1575699.52
8,2017-06,421923.37,1997622.89
9,2017-07,481604.52,2479227.41


## 9. Repeat customer rate
What percentage of customers come back and order again?

In [27]:
query = """
WITH customer_order_counts AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS num_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN num_orders = 1 THEN 1 ELSE 0 END) AS one_time_customers,
    SUM(CASE WHEN num_orders > 1 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(100.0 * SUM(CASE WHEN num_orders > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_customer_pct
FROM customer_order_counts;
"""

pd.read_sql(query, conn)

,total_customers,one_time_customers,repeat_customers,repeat_customer_pct
0,93358,90557,2801,3.0


## 10. Top seller per product category
Which seller generates the most revenue within each category?

In [29]:
query = """
WITH seller_category_revenue AS (
    SELECT
        ct.product_category_name_english AS category,
        oi.seller_id,
        ROUND(SUM(oi.price), 2) AS seller_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    WHERE o.order_status = 'delivered'
    GROUP BY category, oi.seller_id
),
ranked_sellers AS (
    SELECT
        category,
        seller_id,
        seller_revenue,
        RANK() OVER (PARTITION BY category ORDER BY seller_revenue DESC) AS revenue_rank
    FROM seller_category_revenue
)
SELECT *
FROM ranked_sellers
WHERE revenue_rank = 1
ORDER BY seller_revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,category,seller_id,seller_revenue,revenue_rank
0,watches_gifts,4869f7a5dfa277a7dca6462dcf3b52b2,198822.03,1
1,office_furniture,7c67e1448b00f6e969d365cea6b010ab,171605.82,1
2,computers,53243585a1d6dc2643021fd1853d8905,163322.76,1
3,bed_bath_table,4a3ca9315b744ce9f8e9374361493884,162206.95,1
4,cool_stuff,7a67c85e85bb2ce8582c35f2203ad736,133408.69,1
5,furniture_decor,1025f0e2d44d7041d6cf58b6550e0bfa,110770.46,1
6,garden_tools,1f50f920176fa81dab994f9023523100,102849.61,1
7,toys,46dc3b2cc0980fb8ec44634e21d2718e,79337.07,1
8,health_beauty,edb1ef5e36e0c8cd84eb3c9b003e486d,79144.90,1
9,stationery,3d871de0142ce09b7081e2b9d1733cb1,78894.80,1


## 11. Payment method behavior
How do customers prefer to pay?

In [31]:
query = """
SELECT
    payment_type,
    COUNT(*) AS num_payments,
    ROUND(AVG(payment_installments), 1) AS avg_installments,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM order_payments
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY num_payments DESC;
"""

pd.read_sql(query, conn)

,payment_type,num_payments,avg_installments,avg_payment_value
0,credit_card,76795,3.5,163.32
1,boleto,19784,1.0,145.03
2,voucher,5775,1.0,65.70
3,debit_card,1529,1.0,142.57


## Conclusion

This analysis covered 96,478 delivered orders across the Olist marketplace. Total revenue reached $13,221,498.11, with an average order value of $137.04.

Revenue grew steadily from a near-zero base in late 2016 to a peak of roughly $977,500 in May 2018, with November 2017 standing out as a clear outlier ($987,765) — likely driven by Black Friday seasonality. Cumulative revenue confirms consistent, compounding growth across the dataset's full timeframe.

**Health & beauty** was the top-grossing product category ($1,233,131.72), narrowly ahead of **watches & gifts** ($1,166,176.98) and **bed, bath & table** ($1,023,434.76).

Delivery performance has real, measurable consequences: 8.11% of orders (7,826) arrived after their estimated delivery date, and late deliveries saw average review scores drop to 2.57 stars, compared to 4.29 stars for on-time orders — a gap of 1.72 stars directly tied to delivery reliability.

**São Paulo (SP)** dominated revenue by state at $5,067,633.16, more than triple the next-closest state (Rio de Janeiro, $1,759,651.13), though SP's average order value ($125.12) was actually the lowest among the top 10 states — smaller, more frequent orders rather than fewer, larger ones.

The most striking finding: of 93,358 unique customers, only 3.0% (2,801) placed more than one order. Customer retention, not acquisition, stands out as the clearest opportunity for the business to improve long-term revenue.

**Credit card** was the dominant payment method (76,795 payments, 70% of all transactions), with an average of 3.5 installments per purchase — consistent with common Brazilian consumer credit habits — while boleto, voucher, and debit card payments were overwhelmingly paid in a single installment.